# CSV a SQL

En este archivo vamos a crear inserts de SQL a partir de datos de un CSV.

Vamos a seguir varios pasos:

+ Instalar dependencias nuevas.
+ Llenar tablas fuertes
+ Llenar tablas débiles

Importación de librerías

In [4]:
import pandas as pd

Ruta global y lectura del dataset

In [44]:
ds_cotizaciones = pd.read_csv('Cotizaciones.csv', sep=',', encoding='latin1')
ds_proyectos = pd.read_csv('Proyectos.csv', sep=',', encoding='latin1')
ds_servicios = pd.read_csv('Servicios.csv', sep=',', encoding='latin1', dtype={'codigo': str})
ds_rubros = pd.read_csv('Rubros.csv', sep=',', encoding='latin1', dtype={'codigo': str})
ds_subRubros = pd.read_csv('SubRubro.csv', sep=',', encoding='latin1', dtype={'codigo': str})
ds_partidas = pd.read_csv('Partidas.csv', sep=',', encoding='latin1', thousands='.', decimal=',' , dtype={'numero': str})
ds_tiposRecursos = pd.read_csv('Tipos_Recursos.csv', sep=',', encoding='latin1', dtype={'codigo': str})
ds_clasesRecursos = pd.read_csv('ClasesRecurso.csv', sep=',', encoding='latin1', dtype={'codigo': str})
ds_subclasesRecursos = pd.read_csv('SubclasesRecurso.csv', sep=',', encoding='latin1', dtype={'codigo': str})
ds_recursos = pd.read_csv('Recursos.csv', sep=',', encoding='latin1', dtype={'codigo': str})

ds_cotizaciones = ds_cotizaciones.dropna(how='all')
ds_proyectos = ds_proyectos.dropna(how='all')
ds_servicios = ds_servicios.dropna(how='all')
ds_rubros = ds_rubros.dropna(how='all')
ds_subRubros = ds_subRubros.dropna(how='all')
ds_partidas = ds_partidas.dropna(how='all')
ds_tiposRecursos = ds_tiposRecursos.dropna(how='all')
ds_clasesRecursos = ds_clasesRecursos.dropna(how='all')
ds_subclasesRecursos = ds_subclasesRecursos.dropna(how='all')
ds_recursos = ds_recursos.dropna(how='all')

ds_cotizaciones = ds_cotizaciones.rename(columns={
    'Condiciï¿½n_Comercial': 'Condicion_Comercial',
    'Excepciï¿½n': 'Excepcion',
    'Restricciï¿½n': 'Restriccion'
})
ds_recursos['cambio us'] = ds_recursos['cambio us'].astype(str).str.replace('S/', '', regex=False).str.strip()
ds_recursos['cambio us'] = pd.to_numeric(ds_recursos['cambio us'], errors='coerce')

ds_recursos['precio unitario soles'] = pd.to_numeric(ds_recursos['precio unitario soles'].astype(str).str.replace(',', '.'), errors='coerce')
ds_recursos['precio unitario dolares'] = pd.to_numeric(ds_recursos['precio unitario dolares'].astype(str).str.replace(',', '.'), errors='coerce')



Lista todos los jugadores

In [48]:
ds_proyectos

,id,nombre,nombre_Coordinador,apellido_Coordinador,nombre_Cliente,apellido_Cliente
0,1.0,II.EE. DEL EDIFICIO DE OFICINAS,Carlos,Gonzales,Ricardo,Villanueva


In [47]:
ds_partidas[['descripcion', 'precio unitario']]

,descripcion,precio unitario
0,TABLERO GENERAL TG,19209.40
1,TABLERO SERVICIOS GENERALES TSG,13406.18
2,TABLERO TG-AF,28122.62
3,TABLERO ASCENSOR LIMPIO - AZOTEA T-AL,1132.17
4,TABLERO ASCENSOR LIMPIO - AZOTEA T-AS,1132.17
...,...,...
297,NaN,NaN
298,NaN,NaN
299,NaN,NaN
300,NaN,NaN


## Generación de Inserts

### Coordinadores

```sql
CREATE TABLE coordinadores (
  id INTEGER NOT NULL PRIMARY KEY AUTOINCREMENT,
  nombre VARCHAR(40) NOT NULL,
  apellido VARCHAR(40) NOT NULL
);

In [50]:
tmp = ds_proyectos[['nombre_Coordinador', 'apellido_Coordinador']].dropna().drop_duplicates()
text = '-- migrate:up \n\n'
id = 1
coordinadores_dicc = {}

for index, row in tmp.iterrows():
    nombre = str(row['nombre_Coordinador']).replace("'", "''")
    apellido = str(row['apellido_Coordinador']).replace("'", "''")
    
    text += f"INSERT INTO coordinadores (id, nombre, apellido) VALUES ({id}, '{nombre}', '{apellido}');\n"
    
    coordinadores_dicc[(row['nombre_Coordinador'], row['apellido_Coordinador'])] = id
    
    id += 1

text += '\n-- migrate:down \n\nDELETE FROM coordinadores;'

with open('inserts_coordinadores.sql', 'w', encoding='utf-8') as archivo:
    archivo.write(text)


### Tipos_unidades

```sql
CREATE TABLE tipos_unidades (
  id INTEGER NOT NULL PRIMARY KEY AUTOINCREMENT,
  simbolo VARCHAR(5) NOT NULL
);
```

In [52]:
tmp = ds_recursos['tipo unidad']
tipos = sorted(list(tmp.unique()))

text = '-- migrate:up \n\n'
id = 1

tipos_dicc = {}

for simbolo in tipos:
  text = text + f"INSERT INTO tipos_unidades (id, simbolo) VALUES ({id}, '{simbolo}');\n"
  tipos_dicc[simbolo] = id
  id = id + 1
text = text + '\n-- migrate:down \n\nDELETE FROM tipos_unidades;'
with open('inserts_tipos_unidades.sql', 'w') as archivo:
  archivo.write(text)

### Tipos_recursos

```sql
CREATE TABLE tipos_recursos (
  id INTEGER NOT NULL PRIMARY KEY AUTOINCREMENT,
  nombre VARCHAR(30) NOT NULL,
  codigo VARCHAR(30) NOT NULL
);
```

In [53]:
tmp = ds_tiposRecursos[['codigo', 'nombre']].dropna().drop_duplicates()

text = '-- migrate:up \n\n'
id = 1
tipos_recursos_dicc = {} 

for index, row in tmp.iterrows():
    codigo = row['codigo']
    nombre = row['nombre']
    
    text += f"INSERT INTO tipos_recursos (id, codigo, nombre) VALUES ({id}, '{codigo}', '{nombre}');\n"
    
    tipos_recursos_dicc[(codigo, nombre)] = id    
    id += 1

text += '\n-- migrate:down \n\nDELETE FROM tipos_recursos;'

with open('inserts_tipos_recursos.sql', 'w', encoding='utf-8') as archivo:
    archivo.write(text)

### Marcas

```sql
CREATE TABLE marcas (
  id INTEGER NOT NULL PRIMARY KEY AUTOINCREMENT,
  nombre VARCHAR(30)
);
```

In [55]:
tmp = ds_recursos['marca'].dropna()
marcas = sorted(list(tmp.unique()))

text = '-- migrate:up \n\n'
id = 1

marcas_dicc = {}

for nombre in marcas:
  text = text + f"INSERT INTO marcas (id, nombre) VALUES ({id}, '{nombre}');\n"
  marcas_dicc[nombre] = id
  id = id + 1
text = text + '\n-- migrate:down \n\nDELETE FROM marcas;'
with open('inserts_marcas.sql', 'w') as archivo:
  archivo.write(text)

### Servicios

```sql
CREATE TABLE servicios (
  id INTEGER NOT NULL PRIMARY KEY AUTOINCREMENT,
  nombre VARCHAR(40) NOT NULL,
  codigo VARCHAR(30) NOT NULL
  
);
```

In [56]:
tmp = ds_servicios[['codigo', 'nombre']].dropna().drop_duplicates()

text = '-- migrate:up \n\n'
id = 1
servicios_dicc = {} 

for index, row in tmp.iterrows():
    codigo = row['codigo']
    nombre = row['nombre']
    
    text += f"INSERT INTO servicios (id, codigo, nombre) VALUES ({id}, '{codigo}', '{nombre}');\n"
    
    servicios_dicc[(codigo, nombre)] = id    
    id += 1

text += '\n-- migrate:down \n\nDELETE FROM servicios;'

with open('inserts_servicios.sql', 'w', encoding='utf-8') as archivo:
    archivo.write(text)

### Clientes

```sql
CREATE TABLE clientes (
  id INTEGER NOT NULL PRIMARY KEY AUTOINCREMENT,
  nombre VARCHAR(40) NOT NULL,
  apellidos VARCHAR(40) NOT NULL
);
```

In [60]:
tmp = ds_proyectos[['nombre_Cliente', 'apellido_Cliente']].dropna().drop_duplicates()
text = '-- migrate:up \n\n'
id = 1
clientes_dicc = {}

for index, row in tmp.iterrows():
    nombre = str(row['nombre_Cliente']).replace("'", "''")
    apellido = str(row['apellido_Cliente']).replace("'", "''")
    
    text += f"INSERT INTO clientes (id, nombre, apellidos) VALUES ({id}, '{nombre}', '{apellido}');\n"
    
    clientes_dicc[(row['nombre_Cliente'], row['apellido_Cliente'])] = id
    
    id += 1

text += '\n-- migrate:down \n\nDELETE FROM clientes;'

with open('inserts_clientes.sql', 'w', encoding='utf-8') as archivo:
    archivo.write(text)

### Formas de Pago

```sql
CREATE TABLE formas_de_pago (
  id INTEGER NOT NULL PRIMARY KEY AUTOINCREMENT,
  nombre VARCHAR(30) NOT NULL
);
```

In [57]:
tmp = ds_cotizaciones['Forma_Pago'].dropna()
formas = sorted(list(tmp.unique()))

text = '-- migrate:up \n\n'
id = 1

formas_dicc = {}

for nombre in formas:
  text = text + f"INSERT INTO formas_de_pago (id, nombre) VALUES ({id}, '{nombre}');\n"
  formas_dicc[nombre] = id
  id = id + 1
text = text + '\n-- migrate:down \n\nDELETE FROM formas_de_pago;'
with open('inserts_formas_de_pago.sql', 'w') as archivo:
  archivo.write(text)

### Estados

```sql
CREATE TABLE estados (
  id INTEGER NOT NULL PRIMARY KEY AUTOINCREMENT,
  nombre VARCHAR(30) NOT NULL
);
```

In [59]:
tmp = ds_cotizaciones['Estado'].dropna()
estados = sorted(list(tmp.unique()))

text = '-- migrate:up \n\n'
id = 1

estados_dicc = {}

for nombre in estados:
  text = text + f"INSERT INTO estados (id, nombre) VALUES ({id}, '{nombre}');\n"
  estados_dicc[nombre] = id
  id = id + 1
text = text + '\n-- migrate:down \n\nDELETE FROM estados;'
with open('inserts_estados.sql', 'w') as archivo:
  archivo.write(text)

## Proyectos

```sql
CREATE TABLE proyectos (
  id	INTEGER NOT NULL PRIMARY KEY AUTOINCREMENT,
  nombre VARCHAR(40) NOT NULL,
  coordinador_id INTEGER NOT NULL,
  cliente_id INTEGER NOT NULL,
  FOREIGN KEY (coordinador_id) REFERENCES coordinadores (id),
  FOREIGN KEY (cliente_id) REFERENCES clientes (id)
);
```

In [74]:
tmp_proyectos = ds_proyectos.dropna(how='all')

text = '-- migrate:up \n\n'
id_proyecto = 1  
for index, row in tmp_proyectos.iterrows():
    nombre = str(row['nombre ']).replace("'", "''")
    
    nom_coor = row['nombre_Coordinador']
    ape_coor = row['apellido_Coordinador']
    nom_clie = row['nombre_Cliente']
    ape_clie = row['apellido_Cliente']
    
    coordinador_id = coordinadores_dicc.get((nom_coor, ape_coor), 'NULL')
    cliente_id = clientes_dicc.get((nom_clie, ape_clie), 'NULL')
    
    text += f"INSERT INTO proyectos (id, nombre, coordinador_id, cliente_id) VALUES ({id}, '{nombre}', {coordinador_id}, {cliente_id});\n"
    
    id += 1

text += '\n-- migrate:down \n\nDELETE FROM proyectos;'

with open('inserts_proyectos.sql', 'w', encoding='utf-8') as archivo:
    archivo.write(text)

## Cotizaciones

```sql
CREATE TABLE cotizaciones (
  id	INTEGER NOT NULL PRIMARY KEY AUTOINCREMENT,
  numero VARCHAR(8) NOT NULL,
  porcentaje_de_gastos FLOAT NOT NULL,
  porcentaje_de_utilidad FLOAT NOT NULL,
  fecha DATE NOT NULL,
  porcentaje_igv FLOAT NOT NULL,
  total FLOAT NOT NULL,
  monto_igv FLOAT NOT NULL,
  monto_utilidad FLOAT NOT NULL,
  monto_gasto_general FLOAT NOT NULL,
  costo_directo FLOAT NOT NULL,
  subtotal FLOAT NOT NULL,
  alcance VARCHAR(100) NOT NULL,
  condicion_comercial VARCHAR(300) NOT NULL,
  excepcion VARCHAR(300) NOT NULL,
  restriccion VARCHAR(300) NOT NULL,
  proyecto_id INTEGER NOT NULL,
  estado_id INTEGER NOT NULL,
  forma_pago_id INTEGER NOT NULL,
  FOREIGN KEY (proyecto_id) REFERENCES proyectos (id),
  FOREIGN KEY (estado_id) REFERENCES estados (id),
  FOREIGN KEY (forma_pago_id) REFERENCES formas_de_pago (id)
);
```

In [71]:
tmp_cotizaciones = ds_cotizaciones.dropna(how='all')
tmp_cotizaciones.columns = tmp_cotizaciones.columns.str.strip()


estados_dicc = {'Pendiente': 1, 'Aprobado': 2, 'Rechazado': 3}
formas_pago_dicc = {'Adelanto y Sa': 1, 'Contado': 2, 'Crédito': 3}

text = '-- migrate:up \n\n'
id_cotizacion = 1
for index, row in tmp_cotizaciones.iterrows():
    numero = str(row['Numero']).replace("'", "''")
    alcance = str(row['Alcance']).replace("'", "''")
    
    condicion = str(row['Condicion_Comercial']).replace("'", "''")
    excepcion = str(row['Excepcion']).replace("'", "''")
    restriccion = str(row['Restriccion']).replace("'", "''")
    
    porc_gastos = row['Porcentaje_Gastos']
    porc_utilidad = row['Porcentaje_Utilidad']
    fecha = row['fecha']
    porc_igv = row['Porcentaje_IGV']
    total = row['Total']
    monto_igv = row['Monto_IGV']
    monto_utilidad = row['Monto_Utilidad'] 
    monto_gasto = row['Monto_Gasto']
    costo_directo = row['Costo_Directo'] 
    subtotal = row['Subtotal']
    
    proyecto_id = int(row['Proyecto_id']) if pd.notna(row['Proyecto_id']) else 'NULL'
    estado_id = estados_dicc.get(row['Estado'], 'NULL')
    forma_pago_id = formas_dicc.get(row['Forma_Pago'], 'NULL')
    
    text += f"""INSERT INTO cotizaciones (
        id, numero, porcentaje_de_gastos, porcentaje_de_utilidad, fecha, porcentaje_igv, total, 
        monto_igv, monto_utilidad, monto_gasto_general, costo_directo, subtotal, alcance, 
        condicion_comercial, excepcion, restriccion, proyecto_id, estado_id, forma_pago_id
    ) VALUES (
        {id_cotizacion}, '{numero}', {porc_gastos}, {porc_utilidad}, '{fecha}', {porc_igv}, {total}, 
        {monto_igv}, {monto_utilidad}, {monto_gasto}, {costo_directo}, {subtotal}, '{alcance}', 
        '{condicion}', '{excepcion}', '{restriccion}', {proyecto_id}, {estado_id}, {forma_pago_id}
    );\n"""
    
    id_cotizacion += 1

text += '\n-- migrate:down \n\nDELETE FROM cotizaciones;'

with open('inserts_cotizaciones.sql', 'w', encoding='utf-8') as archivo:
    archivo.write(text)

## Rubros

```sql
CREATE TABLE rubros (
  id	INTEGER NOT NULL PRIMARY KEY AUTOINCREMENT,
  nombre VARCHAR(30),
  codigo VARCHAR(30) NOT NULL,  
  servicio_id INTEGER NOT NULL,
  FOREIGN KEY (servicio_id) REFERENCES servicios (id)
);
```

In [73]:
tmp_rubros = ds_rubros.dropna(how='all')
tmp_rubros.columns = tmp_rubros.columns.str.strip()
text = '-- migrate:up \n\n'
id_rubro = 1
rubros_dicc = {}

for index, row in tmp_rubros.iterrows():
    codigo = str(row['codigo']).strip()
    nombre = str(row['nombre']).replace("'", "''").strip()
    servicio_id = int(row['Servicio_id']) if pd.notna(row['Servicio_id']) else 'NULL'
    rubros_dicc[(codigo, nombre)] = id_rubro
    text += f"INSERT INTO rubros (id, nombre, codigo, servicio_id) VALUES ({id_rubro}, '{nombre}', '{codigo}', {servicio_id});\n" 
    id_rubro += 1

text += '\n-- migrate:down \n\nDELETE FROM rubros;'
with open('inserts_rubros.sql', 'w', encoding='utf-8') as archivo:
    archivo.write(text)

## Sub_Rubros

```sql
CREATE TABLE sub_rubros (
  id	INTEGER NOT NULL PRIMARY KEY AUTOINCREMENT,
  nombre VARCHAR(30),
  codigo VARCHAR(30) NOT NULL,
  rubro_id INTEGER NOT NULL,
  FOREIGN KEY (rubro_id) REFERENCES rubros (id)
);

```

In [ ]:
tmp_sub_rubros = ds_subRubros.dropna(how='all')
tmp_sub_rubros.columns = tmp_sub_rubros.columns.str.strip()

text = '-- migrate:up \n\n'
id_sub_rubro = 1
sub_rubros_dicc = {}

for index, row in tmp_sub_rubros.iterrows():
    
    if pd.isna(row['codigo']):
        codigo_sql = "''"
    else:
        codigo_sql = f"'{str(row['codigo']).strip()}'"
        
    if pd.isna(row['nombre']):
        nombre_sql = "NULL"
    else:
        nombre_limpio = str(row['nombre']).replace("'", "''").replace("Ö", "Í").strip()
        nombre_sql = f"'{nombre_limpio}'"
    
    rubro_id = int(row['Rubro_id'])
    
    if pd.notna(row['codigo']):
        codigo_clean = str(row['codigo']).strip()
        nombre_clean = str(row['nombre']).strip() if pd.notna(row['nombre']) else ''
        sub_rubros_dicc[(codigo_clean, nombre_clean)] = id_sub_rubro
    

    text += f"INSERT INTO sub_rubros (id, nombre, codigo, rubro_id) VALUES ({id}, {nombre_sql}, {codigo_sql}, {rubro_id});\n"
    
    id += 1

text += '\n-- migrate:down \n\nDELETE FROM sub_rubros;'

with open('inserts_sub_rubros.sql', 'w', encoding='utf-8') as archivo:
    archivo.write(text)

## Partidas

```sql
CREATE TABLE partidas (
  id	INTEGER NOT NULL PRIMARY KEY AUTOINCREMENT,
  descripcion VARCHAR(100) NOT NULL,
  codigo VARCHAR(30) NOT NULL,
  sub_rubro_id INTEGER NOT NULL,    
  FOREIGN KEY (sub_rubro_id) REFERENCES sub_rubros (id)
);
```

In [ ]:
tmp_partidas_source = ds_partidas.dropna(how='all')
tmp_partidas_source.columns = tmp_partidas_source.columns.str.strip()

tmp_partidas_source = tmp_partidas_source.dropna(subset=['Subrubro_id'])

text_partidas = '-- migrate:up \n\n'
id_partida = 1

partidas_dicc = {}

for index, row in tmp_partidas_source.iterrows():
    codigo = str(row['numero']).strip()
    descripcion = str(row['descripcion']).replace("'", "''").strip()
    sub_rubro_id = int(row['Subrubro_id'])
    partidas_dicc[codigo] = id_partida
    text_partidas += f"INSERT INTO partidas (id, descripcion, codigo, sub_rubro_id) VALUES ({id_partida}, '{descripcion}', '{codigo}', {sub_rubro_id});\n"
    
    id_partida += 1

text_partidas += '\n-- migrate:down \n\nDELETE FROM partidas;'

with open('inserts_partidas.sql', 'w', encoding='utf-8') as archivo:
    archivo.write(text_partidas)

## Proyectos_Partidas

```sql
CREATE TABLE proyectos_partida (
  costo_directo FLOAT,
  metrado FLOAT NOT NULL,
  precio_unitario FLOAT NOT NULL,
  proyecto_id INTEGER NOT NULL,
  partida_id INTEGER NOT NULL,
  FOREIGN KEY (proyecto_id) REFERENCES proyectos (id),
  FOREIGN KEY (partida_id) REFERENCES partidas (id)
);
```

In [ ]:

text_asociativa = '-- migrate:up \n\n'
PROYECTO_ID_CONSTANTE = 1 

for index, row in tmp_partidas_source.iterrows():
    codigo_partida = str(row['numero']).strip()
    partida_id = partidas_dicc.get(codigo_partida)
    
    metrado = float(row['metrado']) if pd.notna(row['metrado']) else 0.0
    precio_unitario = float(row['precio unitario']) if pd.notna(row['precio unitario']) else 0.0
    
    costo_directo = float(row.iloc[4]) if pd.notna(row.iloc[4]) else 0.0
    
    text_asociativa += f"""INSERT INTO proyectos_partida (
        proyecto_id, partida_id, metrado, precio_unitario, costo_directo
    ) VALUES (
        {PROYECTO_ID_CONSTANTE}, {partida_id}, {metrado}, {precio_unitario}, {costo_directo}
    );\n"""

text_asociativa += '\n-- migrate:down \n\nDELETE FROM proyectos_partida;'

with open('inserts_proyectos_partidas.sql', 'w', encoding='utf-8') as archivo:
    archivo.write(text_asociativa)

## Clases_Recursos

```sql
CREATE TABLE clases_recursos (
  id	INTEGER NOT NULL PRIMARY KEY AUTOINCREMENT,
  nombre VARCHAR(30),
  codigo VARCHAR(30) NOT NULL,
  tipo_recurso_id INTEGER NOT NULL,
  FOREIGN KEY (tipo_recurso_id) REFERENCES tipos_recursos (id)
);
```

In [76]:
tmp_clases = ds_clasesRecursos.dropna(how='all')
tmp_clases.columns = tmp_clases.columns.str.strip()

text = '-- migrate:up \n\n'
id_clase = 1

clases_recursos_dicc = {}

for index, row in tmp_clases.iterrows():
    codigo = str(row['codigo']).strip()
    nombre = str(row['nombre']).replace("'", "''")
    nombre = nombre.replace("Ö", "Í").replace("µ", "Á").strip()
    
    tipo_recurso_id = int(row['tipo_recurso_id'])
    
    clases_recursos_dicc[(codigo, nombre)] = id_clase
    
    text += f"INSERT INTO clases_recursos (id, nombre, codigo, tipo_recurso_id) VALUES ({id_clase}, '{nombre}', '{codigo}', {tipo_recurso_id});\n"
    
    id_clase += 1

text += '\n-- migrate:down \n\nDELETE FROM clases_recursos;'

with open('inserts_clases_recursos.sql', 'w', encoding='utf-8') as archivo:
    archivo.write(text)

## Subclases_Recursos

```sql
CREATE TABLE subclases_recursos (
  id	INTEGER NOT NULL PRIMARY KEY AUTOINCREMENT,
  nombre VARCHAR(30),
  codigo VARCHAR(30) NOT NULL, 
  clase_recurso_id INTEGER NOT NULL,
  FOREIGN KEY (clase_recurso_id) REFERENCES clases_recursos (id)
);
```

In [77]:
tmp_subclases = ds_subclasesRecursos.dropna(how='all')
tmp_subclases.columns = tmp_subclases.columns.str.strip()

text = '-- migrate:up \n\n'
id_subclase = 1

subclases_recursos_dicc = {}

for index, row in tmp_subclases.iterrows():
    
    if pd.isna(row['codigo']):
        codigo_sql = "''"
    else:
        cod_limpio = str(int(row['codigo'])) if isinstance(row['codigo'], (int, float)) else str(row['codigo'])
        codigo_sql = f"'{cod_limpio.strip()}'"
        
    if pd.isna(row['nombre']):
        nombre_sql = "NULL"
    else:
        nombre_limpio = str(row['nombre']).replace("'", "''").replace("Ö", "Í").strip()
        nombre_sql = f"'{nombre_limpio}'"
        
    clase_recurso_id = int(row['Clase_Recurso_id'])
    
    if pd.notna(row['codigo']):
        cod_clean = str(int(row['codigo'])) if isinstance(row['codigo'], (int, float)) else str(row['codigo'])
        nom_clean = str(row['nombre']).replace("Ö", "Í").strip() if pd.notna(row['nombre']) else ''
        subclases_recursos_dicc[(cod_clean.strip(), nom_clean)] = id_subclase
        
    text += f"INSERT INTO subclases_recursos (id, nombre, codigo, clase_recurso_id) VALUES ({id_subclase}, {nombre_sql}, {codigo_sql}, {clase_recurso_id});\n"
    
    id_subclase += 1

text += '\n-- migrate:down \n\nDELETE FROM subclases_recursos;'

with open('inserts_subclases_recursos.sql', 'w', encoding='utf-8') as archivo:
    archivo.write(text)

## Recursos

```sql
CREATE TABLE recursos (
  id	INTEGER NOT NULL PRIMARY KEY AUTOINCREMENT,
  codigo VARCHAR(30) NOT NULL,
  precio_unitario_soles FLOAT NOT NULL,
  precio_unitario_dolares FLOAT NOT NULL,
  cambio FLOAT NOT NULL,
  descripcion VARCHAR(100) NOT NULL,
  marca_id INTEGER NOT NULL,
  subclase_recurso_id INTEGER NOT NULL,
  tipo_unidad_id INTEGER NOT NULL,
  FOREIGN KEY (marca_id) REFERENCES marcas (id),
  FOREIGN KEY (subclase_recurso_id) REFERENCES subclases_recursos (id),
  FOREIGN KEY (tipo_unidad_id) REFERENCES tipos_unidades (id)
);
```

In [ ]:

text_recursos = '-- migrate:up \n\n'
id_recurso = 1

recursos_dicc = {}

for index, row in ds_recursos.iterrows():
    codigo = str(row['codigo']).strip()
    descripcion = str(row['descripcion']).replace("'", "''").replace("cï¿½n", "ón").replace("eï¿½n", "ón").replace("Peï¿½n", "Peón").strip()
    precio_soles = float(row['precio unitario soles']) if pd.notna(row['precio unitario soles']) else 0.0
    precio_dolares = float(row['precio unitario dolares']) if pd.notna(row['precio unitario dolares']) else 0.0
    cambio = float(row['cambio us']) if pd.notna(row['cambio us']) else 3.30
    subclase_id = int(row['subclaseRecurso_id'])
    nom_marca = str(row['marca']).strip() if pd.notna(row['marca']) else ''
    marca_id = marcas_dicc.get(nom_marca, 1) 
    nom_unidad = str(row['tipo unidad']).strip() if pd.notna(row['tipo unidad']) else ''
    tipo_unidad_id = tipos_dicc.get(nom_unidad, 1)
    recursos_dicc[codigo] = id_recurso
    
    text_recursos += f"""INSERT INTO recursos (
        id, codigo, precio_unitario_soles, precio_unitario_dolares, cambio, descripcion, marca_id, subclase_recurso_id, tipo_unidad_id
    ) VALUES (
        {id_recurso}, '{codigo}', {precio_soles}, {precio_dolares}, {cambio}, '{descripcion}', {marca_id}, {subclase_id}, {tipo_unidad_id}
    );\n"""
    
    id_recurso += 1

text_recursos += '\n-- migrate:down \n\nDELETE FROM recursos;'

with open('inserts_recursos.sql', 'w', encoding='utf-8') as archivo:
    archivo.write(text_recursos)

## Partidas_Recursos

```sql
CREATE TABLE partidas_recursos (
  partida_id INTEGER NOT NULL,
  recurso_id INTEGER NOT NULL,
  FOREIGN KEY (partida_id) REFERENCES partidas (id),
  FOREIGN KEY (recurso_id) REFERENCES recursos (id)
);

```

In [105]:
import pandas as pd
df_mapeo_validado = pd.read_excel('Partidas_Recursos.xlsx')

recursos_dicc_seguro = {}
for idx_r, row_r in ds_recursos.iterrows():
    r_id = row_r['id']
    r_codigo = str(row_r['codigo']).strip()
    
    if r_codigo.endswith('.0'):
        r_codigo = r_codigo[:-2]
    if len(r_codigo) < 6 and r_codigo.isdigit():
        r_codigo = r_codigo.zfill(6)
        
    recursos_dicc_seguro[r_codigo] = int(r_id)

text_sql = '-- migrate:up \n\n'
insert_count = 0
filas_omitidas = 0

for _, row in df_mapeo_validado.iterrows():
    partida_id_raw = str(row['Partida_ID_BD']).strip()
    recurso_cod_raw = str(row['Recurso_Cod']).strip()
    
    if recurso_cod_raw.endswith('.0'):
        recurso_cod_raw = recurso_cod_raw[:-2]
    if len(recurso_cod_raw) < 6 and recurso_cod_raw.isdigit():
        recurso_cod_raw = recurso_cod_raw.zfill(6)
        
    if partida_id_raw.isdigit():
        p_id = int(partida_id_raw)
        r_id = recursos_dicc_seguro.get(recurso_cod_raw)
        
        if r_id:
            text_sql += f"INSERT INTO partidas_recursos (partida_id, recurso_id) VALUES ({p_id}, {r_id});\n"
            insert_count += 1
        else:
            filas_omitidas += 1
    else:
        filas_omitidas += 1

text_sql += '\n-- migrate:down \n\nDELETE FROM partidas_recursos;'

with open('inserts_partidas_recursos_final.sql', 'w', encoding='utf-8') as archivo:
    archivo.write(text_sql)
